In [11]:
# !pip install -q pandas scikit-learn joblib nltk

In [12]:
# imports
import pandas as pd
import numpy as np
import re
import joblib
import nltk
import warnings

warnings.filterwarnings("ignore")

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, classification_report, f1_score

nltk.download("stopwords")
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
# Load dataset CSVs directly from content folder
filename_training = "twitter_training.csv"
filename_validation = "twitter_validation.csv"

# Load twitter_training.csv for the main analysis, assuming no header in the file itself
df = pd.read_csv(filename_training, header=None)
# Manually assign column names as per the dataset description
df.columns = ['Tweet ID', 'Entity', 'Sentiment', 'Tweet content']

print("\nDataset shape ('twitter_training.csv'):", df.shape)
print(df.head())

# Note: twitter_validation.csv is assumed to be in the content folder as well, but 'df' currently holds twitter_training.csv


Dataset shape ('twitter_training.csv'): (53827, 4)
   Tweet ID       Entity Sentiment  \
0      2401  Borderlands  Positive   
1      2401  Borderlands  Positive   
2      2401  Borderlands  Positive   
3      2401  Borderlands  Positive   
4      2401  Borderlands  Positive   

                                       Tweet content  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  


In [18]:
# rename and drop colunms
print("\nColumns before renaming:")
print(df.columns)

# The Kaggle dataset 'twitter_training.csv' has columns:
# 'Tweet ID', 'Entity', 'Sentiment', 'Tweet content'
# We need 'text' and 'sentiment' for the existing pipeline.

# Drop unnecessary columns if they exist
columns_to_drop = [col for col in ["Tweet ID", "Entity"] if col in df.columns]
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)

df = df.rename(columns={
    "Tweet content": "text",
    "Sentiment": "sentiment"
})

print("\nColumns after renaming:")
print(df.columns)
print(df.head())

# clean labels
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()

valid_labels = ["negative", "neutral", "positive"]
df = df[df["sentiment"].isin(valid_labels)]

print("\nLabel counts:")
print(df["sentiment"].value_counts())

# Encode labels
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

reverse_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

df["label"] = df["sentiment"].map(label_map)

# text cleaning
def clean_text(text):
    text = str(text).lower()

    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = [w for w in text.split() if w not in STOPWORDS]

    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_text)

print("\nSample cleaned text:")
print(df["clean_text"].head())


Columns before renaming:
Index(['Tweet ID', 'Entity', 'Sentiment', 'Tweet content'], dtype='object')

Columns after renaming:
Index(['sentiment', 'text'], dtype='object')
  sentiment                                               text
0  Positive  im getting on borderlands and i will murder yo...
1  Positive  I am coming to the borders and I will kill you...
2  Positive  im getting on borderlands and i will kill you ...
3  Positive  im coming on borderlands and i will murder you...
4  Positive  im getting on borderlands 2 and i will murder ...

Label counts:
sentiment
positive    16080
negative    14911
neutral     12882
Name: count, dtype: int64

Sample cleaned text:
0    im getting borderlands murder
1              coming borders kill
2      im getting borderlands kill
3     im coming borderlands murder
4    im getting borderlands murder
Name: clean_text, dtype: object


In [19]:
# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("\nTrain size:", len(X_train))
print("Test size :", len(X_test))

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vectorized shape:", X_train_vec.shape)

# define models
models = {
    "LogisticRegression": LogisticRegression(max_iter=500, n_jobs=-1),
    "LinearSVC": LinearSVC(),
    "MultinomialNB": MultinomialNB()
}

results = {}

best_model = None
best_name = None
best_f1 = -1


Train size: 35098
Test size : 8775
Vectorized shape: (35098, 20000)


In [20]:
# train/evaluate
for name, model in models.items():

    print("\n==============================")
    print("Training:", name)

    model.fit(X_train_vec, y_train)

    pred = model.predict(X_test_vec)

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="macro")

    print("Accuracy :", round(acc,4))
    print("Macro F1 :", round(f1,4))
    print(classification_report(y_test, pred, target_names=valid_labels))

    results[name] = {
        "accuracy": acc,
        "f1": f1
    }

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_name = name


Training: LogisticRegression
Accuracy : 0.8342
Macro F1 : 0.8329
              precision    recall  f1-score   support

    negative       0.81      0.87      0.84      2982
     neutral       0.85      0.78      0.82      2577
    positive       0.85      0.84      0.84      3216

    accuracy                           0.83      8775
   macro avg       0.84      0.83      0.83      8775
weighted avg       0.84      0.83      0.83      8775


Training: LinearSVC
Accuracy : 0.8896
Macro F1 : 0.8894
              precision    recall  f1-score   support

    negative       0.92      0.88      0.90      2982
     neutral       0.89      0.87      0.88      2577
    positive       0.86      0.91      0.89      3216

    accuracy                           0.89      8775
   macro avg       0.89      0.89      0.89      8775
weighted avg       0.89      0.89      0.89      8775


Training: MultinomialNB
Accuracy : 0.7838
Macro F1 : 0.7784
              precision    recall  f1-score   support


In [21]:
# the best model
print("BEST MODEL:", best_name)
print("BEST MACRO F1:", round(best_f1,4))

# save files
joblib.dump(best_model, "best_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("\nSaved:")
print("- best_model.pkl")
print("- vectorizer.pkl")


BEST MODEL: LinearSVC
BEST MACRO F1: 0.8894

Saved:
- best_model.pkl
- vectorizer.pkl


In [22]:
# downloads
files.download("best_model.pkl")
files.download("vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# quick test
samples = [
    "I love this phone so much",
    "It is okay nothing special",
    "Worst service ever"
]

X_demo = vectorizer.transform(samples)
preds = best_model.predict(X_demo)

print("\nDemo Predictions:")
for text, pred in zip(samples, preds):
    print(text, "->", reverse_map[pred])


Demo Predictions:
I love this phone so much -> positive
It is okay nothing special -> neutral
Worst service ever -> negative
